# Autoresearch-GLM Experiment Analysis

Analysis of autonomous feature-search results on the TaiwanCredit benchmark from `results.tsv`.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Expected columns: commit, val_auc, status, description
df = pd.read_csv('results.tsv', sep='\t')
df['val_auc'] = pd.to_numeric(df['val_auc'], errors='coerce')
df['status'] = df['status'].astype(str).str.strip().str.upper()

if 'num_features' in df.columns:
    df['num_features'] = pd.to_numeric(df['num_features'], errors='coerce')

print(f'Total experiments: {len(df)}')
print(f'Columns: {list(df.columns)}')
df.head(10)


In [ ]:
counts = df['status'].value_counts()
print('Experiment outcomes:')
print(counts.to_string())

n_keep = counts.get('KEEP', 0)
n_discard = counts.get('DISCARD', 0)
n_crash = counts.get('CRASH', 0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f'\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}')
print(f'Crashes: {n_crash}')


In [ ]:
kept = df[df['status'] == 'KEEP'].copy()
print(f'KEPT experiments ({len(kept)} total):\n')
for i, row in kept.iterrows():
    extra = ''
    if 'num_features' in row and pd.notna(row['num_features']):
        extra = f"  features={int(row['num_features'])}"
    print(f"  #{i:3d}  auc={row['val_auc']:.6f}{extra}  {row['description']}")


## Val AUC Over Time

Track how the best kept validation AUC evolves as experiments progress. Higher is better, so the frontier is the running maximum.


In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

valid = df[df['status'] != 'CRASH'].copy().reset_index(drop=True)
baseline_auc = valid.loc[0, 'val_auc']
above = valid[valid['val_auc'] >= baseline_auc - 0.01]

disc = above[above['status'] == 'DISCARD']
ax.scatter(disc.index, disc['val_auc'], c='#cccccc', s=12, alpha=0.5, zorder=2, label='Discarded')

kept_v = above[above['status'] == 'KEEP']
ax.scatter(kept_v.index, kept_v['val_auc'], c='#2ecc71', s=50, zorder=4, label='Kept', edgecolors='black', linewidths=0.5)

kept_mask = valid['status'] == 'KEEP'
kept_idx = valid.index[kept_mask]
kept_auc = valid.loc[kept_mask, 'val_auc']
running_max = kept_auc.cummax()
ax.step(kept_idx, running_max, where='post', color='#27ae60', linewidth=2, alpha=0.7, zorder=3, label='Running best')

for idx, auc in zip(kept_idx, kept_auc):
    desc = str(valid.loc[idx, 'description']).strip()
    if len(desc) > 70:
        desc = desc[:67] + '...'
    ax.annotate(desc, (idx, auc), xytext=(5, 5), textcoords='offset points', fontsize=8, alpha=0.85)

ax.axhline(baseline_auc, color='#34495e', linestyle='--', linewidth=1.5, alpha=0.8, label=f'Baseline ({baseline_auc:.4f})')
ax.set_title('Autoresearch-GLM on TaiwanCredit: validation AUC over experiments')
ax.set_xlabel('Experiment index')
ax.set_ylabel('Validation AUC')
ax.grid(True, alpha=0.2)
ax.legend()
plt.show()


## Summary Statistics


In [ ]:
kept = df[df['status'] == 'KEEP'].copy()
baseline_auc = df.iloc[0]['val_auc']
best_auc = kept['val_auc'].max()
best_row = kept.loc[kept['val_auc'].idxmax()]

print(f'Baseline val_auc:  {baseline_auc:.6f}')
print(f'Best val_auc:      {best_auc:.6f}')
print(f'Total improvement: {best_auc - baseline_auc:+.6f} ({(best_auc - baseline_auc) / baseline_auc * 100:.2f}%)')
print(f"Best experiment:   {best_row['description']}")
print()
print('Cumulative frontier improvements:')
kept_sorted = kept.reset_index()
for _, row in kept_sorted.iterrows():
    print(f"  Experiment #{int(row['index']):3d}: auc={row['val_auc']:.6f}  {row['description']}")


## Top Hits (Kept Experiments by Improvement)


In [ ]:
kept = df[df['status'] == 'KEEP'].copy()
kept['prev_auc'] = kept['val_auc'].shift(1)
kept['delta'] = kept['val_auc'] - kept['prev_auc']
hits = kept.iloc[1:].copy().sort_values('delta', ascending=False)

print(f"{'Rank':>4}  {'Delta':>8}  {'AUC':>10}  Description")
print('-' * 80)
for rank, (_, row) in enumerate(hits.iterrows(), 1):
    print(f"{rank:4d}  {row['delta']:+.6f}  {row['val_auc']:.6f}  {row['description']}")

print(f"\n{'':>4}  {hits['delta'].sum():+.6f}  {'':>10}  TOTAL improvement over baseline")
